In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset,DataLoader
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.metrics import(classification_report,ConfusionMatrixDisplay)

In [ ]:
torch.manual_seed(42)

x,y = make_classification(n_samples=1000,n_features=3,n_informative=3,n_redundant=0,random_state=45)
print(x)
print(x.shape)
print(y)
print(y.shape)

In [ ]:
x_train,x_temp,y_train,y_temp = train_test_split(x,y,test_size=0.30,random_state=42,stratify=y)
x_val,x_test,y_val,y_test = train_test_split(x_temp,y_temp,test_size=0.50,random_state=42,stratify=y_temp)


In [ ]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)
x_val = scaler.transform(x_val)

In [ ]:
x_train_t = torch.tensor(x_train,dtype=torch.float32)
y_train_t = torch.tensor(y_train,dtype=torch.float32).view(-1,1)

x_val_t = torch.tensor(x_val,dtype=torch.float32)
y_val_t = torch.tensor(y_val,dtype=torch.float32).view(-1,1)

x_test_t = torch.tensor(x_test,dtype=torch.float32)
y_test_t = torch.tensor(y_test,dtype=torch.float32).view(-1,1)
print(x_train_t)
print(x_train_t.shape)
print(type(x_train_t))

In [ ]:
train_dataset = TensorDataset(x_train_t,y_train_t)
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True) # batch size mean we get 32 by 32
print(train_loader)                                                 # shuffle mean we need to train model we have to give epoc then each epoc dataset is shuffle
print(type(train_loader))
print(list(train_loader))

In [ ]:
class CustomerPredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(3,8) # hidden layer input 3 and output 8
        self.output = nn.Linear(8,1)#output layer input 8 and output 1
        
    def forward(self,x): # forward pass represent
        x = torch.relu(self.hidden(x))#hidden call inside dunder method
        return self.output(x) # we  return score value not probability after that we call method to that method have sigmoid and combination

model = CustomerPredictor()        


In [ ]:
loss_function = nn.BCEWithLogitsLoss() # calculate probability with losss
optimizer = optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4) # lr and weight_decay is hyperparameter


In [ ]:
epochs = 50
train_losses = []
val_losses = []
CHECKPOINT_PATH = "best_customer_predictor.pt"
best_val_loss = float("inf") # mean we store +infinity value for it when loss find one by one thern (all_loss < +infinity) then assume that new log value

print("Starting Training...")


In [ ]:
for epoch in range(epochs):
    model.train()
    epoch_train_loss = 0.0
    
    for batch_x,batch_y in train_loader:
        optimizer.zero_grad()# previous calculated gradients zero
        logits = model(batch_x)
        loss = loss_function(logits,batch_y) # we give only logit value because loss_function calculate probability with loss
        
        loss.backward()
        optimizer.step() # calculate new weights and bias
        
        epoch_train_loss += loss.item()
    
    avg_train_loss = epoch_train_loss / len(train_loader) #per sample loses
    train_losses.append(avg_train_loss) 
    
    model.eval()
    
    with torch.no_grad(): #stop the backword propagation in validation train
        val_logits = model(x_val_t) #in this section validation test does not train badge vise because it has small samples when that sample count are high then we can use badge vise
        val_loss = loss_function(val_logits,y_val_t)
    val_losses.append(val_loss.item())
    
    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        torch.save(model.state_dict(),CHECKPOINT_PATH) # save to model state dictionary
    
    if(epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | val Loss: {val_loss.item():.4f}")    
           

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(train_losses) + 1), train_losses, label="Training loss")
plt.plot(range(1, len(val_losses) + 1), val_losses, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
try:
    ckpt = torch.load(CHECKPOINT_PATH,map_location="cpu",weights_only=True)
except TypeError:
    ckpt = torch.load(CHECKPOINT_PATH,map_location="cpu")

model.load_state_dict(ckpt) # before this we should include customer predictor but this situation it has above then this situation no need
print(f"Training Complete! Loaded best checkpoint (val loss {best_val_loss:.4f}).")        

In [ ]:
model.eval()
with torch.no_grad():
    test_logits = model(x_test_t)
    test_proba = torch.sigmoid(test_logits).numpy().ravel()
    y_true = y_test_t.numpy().ravel()
    y_pred = (test_proba >= 0.5).astype(int)
print("y_pred",y_pred)


    

In [ ]:
model.eval()
with torch.no_grad():
    probabilities = torch.sigmoid(model(x_test_t))
    y_pred = (probabilities >= 0.5).int().numpy().ravel()


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
plt.show()


In [ ]:
new_customer = [[1.0,2.0,3.0]]
new_customer_t = torch.tensor(scaler.transform(new_customer),dtype=torch.float32)

model.eval()
with torch.no_grad():
    predicted_logit = model(new_customer_t)
    p_buy = torch.sigmoid(predicted_logit).item()
    print(f"predicted probability: {p_buy}")
    

if p_buy > 0.5 :
    print("Customer like")
else:
    print("Customer unlike")        




